###System tables
https://docs.databricks.com/aws/en/admin/system-tables/

In [0]:
import requests

In [0]:
dbx_url='XX.azuredatabricks.net'
metastore_id=dbutils.secrets.get(scope = "", key = "")
pat_token=dbutils.secrets.get(scope = "", key = "")

In [0]:
response = requests.get(f"https://{dbx_url}/api/2.0/unity-catalog/metastores/{metastore_id}/systemschemas", headers={"Authorization": f"Bearer {pat_token}"})
response.text

Enabling system tables

In [0]:
response = requests.put(f"https://{dbx_url}/api/2.0/unity-catalog/metastores/{metastore_id}/systemschemas/access", headers={"Authorization": f"Bearer {pat_token}"})
response

In [0]:
response = requests.put(f"https://{dbx_url}/api/2.0/unity-catalog/metastores/{metastore_id}/systemschemas/query",
                         headers={"Authorization": f"Bearer {pat_token}"})
response

In [0]:
response = requests.put(f"https://{dbx_url}/api/2.0/unity-catalog/metastores/{metastore_id}/systemschemas/lakeflow",
                         headers={"Authorization": f"Bearer {pat_token}"})
response

In [0]:
response = requests.put(f"https://{dbx_url}/api/2.0/unity-catalog/metastores/{metastore_id}/systemschemas/storage",
                         headers={"Authorization": f"Bearer {pat_token}"})
response

##Exploring system tables

In [0]:
%sql
select distinct service_name,action_name from system.access.audit order by service_name,action_name;

In [0]:
%sql
Select request_params.commandText,request_params.executionTime,request_params.clusterId 
 from system.access.audit where  action_name in ('runCommand');

In [0]:
%sql
Select * from system.access.column_lineage --where created_by= '';

In [0]:
%sql
Select * from system.access.table_lineage --where created_by = '';

In [0]:
%sql
Select * from system.billing.list_prices

In [0]:
%sql
Select * from system.billing.usage

In [0]:
%sql
Select * from system.compute.clusters --where owned_by='';


In [0]:
%sql
Select * from system.compute.node_timeline;

In [0]:
%sql
select * from system.information_schema.tables

In [0]:
%sql
select * from system.information_schema.columns

In [0]:
%sql
select * from system.lakeflow.jobs

In [0]:
%sql
select compute.cluster_id,compute.warehouse_id,* from system.query.history 

In [0]:
%sql
select * from system.storage.predictive_optimization_operations_history 

## Dashboard queries

In [0]:
%sql
SELECT
   usage_metadata,U.sku_name,
   round(usage_quantity,2) as usage_quantity,round(U.usage_quantity * P.pricing.effective_list.default,2) AS usage_cost,
   usage_start_time,usage_end_time,U.workspace_id,billing_origin_product 
from system.billing.usage U
JOIN system.billing.list_prices P ON P.sku_name = U.sku_name
WHERE U.usage_end_time >= P.price_start_time
AND (P.price_end_time IS NULL OR U.usage_end_time < P.price_end_time)



In [0]:
%sql
WITH UsageMetadata as (
SELECT
   usage_metadata.cluster_id,usage_metadata,U.sku_name,
   round(usage_quantity,2) as usage_quantity,
   round(U.usage_quantity * P.pricing.effective_list.default,2) AS usage_cost,
   usage_start_time,usage_end_time 
from system.billing.usage U
JOIN system.billing.list_prices P ON P.sku_name = U.sku_name
WHERE U.usage_end_time >= P.price_start_time
AND (P.price_end_time IS NULL OR U.usage_end_time < P.price_end_time)
),
Clusters as (
    SELECT cluster_id,cluster_name,worker_count,
    row_number() OVER(PARTITION BY cluster_id ORDER BY change_time DESC) AS cluster_version,
    owned_by
    FROM system.compute.clusters ),
Commands AS (
Select request_params.commandText as commandText,cast(request_params.executionTime as float) as executionTime,
request_params.clusterId as cluster_id,event_date
FROM system.access.audit where  action_name in ('runCommand')
)    
SELECT U.*,C.cluster_name,C.worker_count,C.owned_by,Q.commandText,Q.executionTime,Q.event_date 
FROM UsageMetadata U
LEFT JOIN Clusters C ON U.cluster_id=C.cluster_id
LEFT JOIN Commands Q ON Q.cluster_id=C.cluster_id
WHERE C.cluster_version=1


In [0]:
%sql
WITH UsageMetadata as (
SELECT
   usage_metadata.warehouse_id,usage_metadata,U.sku_name,
   round(usage_quantity,2) as usage_quantity,
   round(U.usage_quantity * P.pricing.effective_list.default,2) AS usage_cost,
   usage_start_time,usage_end_time 
from system.billing.usage U
JOIN system.billing.list_prices P ON P.sku_name = U.sku_name
WHERE U.usage_end_time >= P.price_start_time
AND (P.price_end_time IS NULL OR U.usage_end_time < P.price_end_time)
),
Warehouses as (
    SELECT warehouse_id,warehouse_name,warehouse_type,warehouse_size 
    FROM system.compute.warehouses),
Queries as (
select compute.warehouse_id as warehouse_id,total_duration_ms/1000 as total_duration_sec ,start_time,statement_text,execution_status,client_application
 from system.query.history)
SELECT U.*,W.warehouse_name,W.warehouse_size,W.warehouse_type,Q.total_duration_sec,Q.start_time,
Q.statement_text,Q.execution_status,Q.client_application
 FROM UsageMetadata U
INNER JOIN Warehouses W ON U.warehouse_id=W.warehouse_id
LEFT JOIN Queries Q ON Q.warehouse_id =W.warehouse_id




In [0]:
%sql
SELECT
   usage_metadata.job_id,J.name as job_name,
   usage_metadata,U.sku_name,
   round(usage_quantity,2) as usage_quantity,round(U.usage_quantity * P.pricing.effective_list.default,2) AS usage_cost,
   usage_start_time,usage_end_time 
from system.billing.usage U
JOIN system.billing.list_prices P ON P.sku_name = U.sku_name
LEFT JOIN system.lakeflow.jobs J ON U.usage_metadata.job_id=J.job_id
WHERE U.usage_end_time >= P.price_start_time
AND (P.price_end_time IS NULL OR U.usage_end_time < P.price_end_time)
AND (usage_metadata.job_id IS NOT NULL)


In [0]:
%sql
SELECT
   U.usage_metadata.dlt_pipeline_id,
   usage_metadata,U.sku_name,
   round(usage_quantity,2) as usage_quantity,round(U.usage_quantity * P.pricing.effective_list.default,2) AS usage_cost,
   usage_start_time,usage_end_time 
from system.billing.usage U
JOIN system.billing.list_prices P ON P.sku_name = U.sku_name
LEFT JOIN system.lakeflow.jobs J ON U.usage_metadata.job_id=J.job_id
WHERE U.usage_end_time >= P.price_start_time
AND (P.price_end_time IS NULL OR U.usage_end_time < P.price_end_time)
AND (U.usage_metadata.dlt_pipeline_id IS NOT NULL)
